# 第四周作业 预测-线性回归
任务背景：本项目提供了一系列学生学习数据，包括学习习惯、出勤率、家长的教育参与程度等。

任务描述：
1. 数据集中存在如School_Type（公立/私立）、Motivation_Level（高/中/低）等分类变量。请对这些分类预测变量进行哑元编码（Dummy Coding） ，将其转换为数值型输入。
2. 选取至少5个与学生个人特征相关的预测变量，以Exam_Score为因变量建立多元线性回归模型。注意评估是否存在多重共线性。
3. 评价模型优劣与泛化性能，并解释模型结果，提出有价值的教育教学建议。

最终提交：程序代码（本文件）。

本任务允许且鼓励你使用AI工具辅助完成，如果使用，在下方AI对话记录板块粘贴对话链接。

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# 读取数据
file_path = "StudentPerformanceFactors.csv"
df = pd.read_csv(file_path)

# 找出分类变量列
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print("原始数据维度：", df.shape)
print("分类变量列：")
for column in categorical_columns:
    print("-", column)

# 对分类变量做哑元编码，drop_first=True 用来减少完全共线性
df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=True, dtype=int)

print("\n哑元编码后的数据维度：", df_encoded.shape)
print("\n哑元编码后新增的部分变量示例：")
encoded_example_columns = [col for col in df_encoded.columns if col not in df.columns]
print(encoded_example_columns[:15])

# 导出哑元编码后的新数据表
df_encoded.to_csv("StudentPerformanceFactors_dummy_encoded.csv", index=False)
print("\n哑元编码后的数据已保存为：StudentPerformanceFactors_dummy_encoded.csv")

print("\n哑元编码后的前5行数据：")
display(df_encoded.head())

# 第二题：建立多元线性回归模型
selected_features = [
    "Hours_Studied",
    "Attendance",
    "Previous_Scores",
    "Tutoring_Sessions",
    "Sleep_Hours",
    "Motivation_Level",
    "Parental_Involvement",
    "Internet_Access",
    "Learning_Disabilities",
]
target = "Exam_Score"

print("\n" + "=" * 60)
print("第二题：多元线性回归建模")
print("选取的自变量：", selected_features)

model_df = df[selected_features + [target]].copy()
X = pd.get_dummies(model_df[selected_features], drop_first=True, dtype=int)
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_train_pred = lr_model.predict(X_train)
y_test_pred = lr_model.predict(X_test)

metrics_df = pd.DataFrame(
    {
        "指标": ["训练集 R^2", "测试集 R^2", "训练集 RMSE", "测试集 RMSE"],
        "数值": [
            r2_score(y_train, y_train_pred),
            r2_score(y_test, y_test_pred),
            np.sqrt(mean_squared_error(y_train, y_train_pred)),
            np.sqrt(mean_squared_error(y_test, y_test_pred)),
        ],
    }
)
metrics_df["数值"] = metrics_df["数值"].round(4)

coef_df = pd.DataFrame(
    {
        "变量": X.columns,
        "回归系数": lr_model.coef_,
    }
).sort_values("回归系数", ascending=False)
coef_df["回归系数"] = coef_df["回归系数"].round(4)

X_with_const = add_constant(X)
vif_df = pd.DataFrame(
    {
        "变量": X_with_const.columns,
        "VIF": [
            variance_inflation_factor(X_with_const.values, i)
            for i in range(X_with_const.shape[1])
        ],
    }
)
vif_df["VIF"] = vif_df["VIF"].round(4)

ols_model = sm.OLS(y, X_with_const).fit()
significance_df = pd.DataFrame(
    {
        "变量": ols_model.params.index,
        "系数": ols_model.params.values,
        "p值": ols_model.pvalues.values,
    }
)
significance_df["系数"] = significance_df["系数"].round(4)
significance_df["p值"] = significance_df["p值"].round(6)

equation_terms = [
    f"({coef:.3f})*{feature}" for feature, coef in zip(X.columns, lr_model.coef_)
]
equation = f"Exam_Score = {lr_model.intercept_:.3f} + " + " + ".join(equation_terms)
equation = equation.replace("+ (-", "- (")

print("\n回归方程（近似写法）：")
print(equation)
print("\n哑变量参考组说明：")
print("- Motivation_Level 的参考组为 High")
print("- Parental_Involvement 的参考组为 High")
print("- Internet_Access 的参考组为 No")
print("- Learning_Disabilities 的参考组为 No")

print("\n模型评估指标：")
display(metrics_df)

print("回归系数表：")
display(coef_df)

print("多重共线性检验（VIF）：")
display(vif_df)

print("系数显著性结果（OLS）：")
display(significance_df)

print(f"总体 R^2: {ols_model.rsquared:.4f}")
print(f"调整后 R^2: {ols_model.rsquared_adj:.4f}")
print("说明：测试集 R^2 越接近训练集 R^2，说明模型泛化能力越稳定。")

# 第三题：评价模型优劣、泛化性能，并提出教育教学建议
significant_df = significance_df[
    (significance_df["变量"] != "const") & (significance_df["p值"] < 0.05)
].copy()
significant_df["绝对系数"] = significant_df["系数"].abs()
important_effects_df = significant_df.sort_values("绝对系数", ascending=False)[
    ["变量", "系数", "p值"]
]

print("\n" + "=" * 60)
print("第三题：模型解释与教育教学建议")

print("\n一、模型优劣评价")
print(
    f"1. 本模型的总体 R^2 为 {ols_model.rsquared:.4f}，调整后 R^2 为 {ols_model.rsquared_adj:.4f}，"
    "说明模型能够解释约 64% 的考试成绩波动，具有较好的解释能力。"
)
print(
    f"2. 训练集 R^2 = {r2_score(y_train, y_train_pred):.4f}，测试集 R^2 = {r2_score(y_test, y_test_pred):.4f}，"
    "两者较为接近，说明模型在新样本上的泛化表现较稳定。"
)
print(
    f"3. 训练集 RMSE = {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}，测试集 RMSE = {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}，"
    "说明模型预测值与真实成绩之间的平均误差大约在 2 分左右。"
)
print(
    f"4. 各自变量的 VIF 大多接近 1，最高约为 {vif_df.loc[vif_df['变量'] != 'const', 'VIF'].max():.4f}，"
    "表明模型不存在明显的多重共线性问题。"
)
print(
    "5. 模型的不足在于：线性回归只能刻画线性关系，且仍有约 35% 左右的成绩波动无法被当前变量解释，"
    "因此结果更适合用于解释和辅助预测，而不是绝对判断。"
)

print("\n二、关键结果解释")
display(important_effects_df)
print("1. Hours_Studied、Attendance、Tutoring_Sessions、Previous_Scores 的系数为正，说明学习投入和既有基础越好，考试成绩通常越高。")
print("2. Internet_Access_Yes 的系数为正，说明具备网络条件的学生平均成绩更高，学习资源可及性较重要。")
print("3. Motivation_Level_Low 和 Motivation_Level_Medium 的系数为负，说明相对于 High 组，学习动机较低的学生成绩更低。")
print("4. Parental_Involvement_Low 和 Parental_Involvement_Medium 的系数为负，说明相对于家长高参与组，家长参与不足会降低学生成绩。")
print("5. Learning_Disabilities_Yes 的系数为负，说明存在学习障碍的学生可能需要更多个性化支持。")
print("6. Sleep_Hours 在本模型中的 p 值较大，未达到显著水平，说明它在线性模型中的单独影响不明显。")

print("\n三、教育教学建议")
print("1. 教师可重点关注学习时长和出勤率较低的学生，尽早进行学习习惯干预。")
print("2. 对基础较弱的学生，可以增加针对性的课后辅导与阶段性反馈，帮助其稳步提升。")
print("3. 学校和教师应通过目标激励、学习反馈和课堂参与设计，提高学生的学习动机。")
print("4. 家校合作非常重要，应加强家长与学校之间的沟通，提升家长的教育参与程度。")
print("5. 对网络条件不足或存在学习障碍的学生，应提供额外资源支持与差异化教学帮助。")
print("6. 在实际教学中，可将本模型作为学生学习预警和教学干预的辅助工具，而不是唯一依据。")

原始数据维度： (6607, 20)
分类变量列：
- Parental_Involvement
- Access_to_Resources
- Extracurricular_Activities
- Motivation_Level
- Internet_Access
- Family_Income
- Teacher_Quality
- School_Type
- Peer_Influence
- Learning_Disabilities
- Parental_Education_Level
- Distance_from_Home
- Gender

哑元编码后的数据维度： (6607, 28)

哑元编码后新增的部分变量示例：
['Parental_Involvement_Low', 'Parental_Involvement_Medium', 'Access_to_Resources_Low', 'Access_to_Resources_Medium', 'Extracurricular_Activities_Yes', 'Motivation_Level_Low', 'Motivation_Level_Medium', 'Internet_Access_Yes', 'Family_Income_Low', 'Family_Income_Medium', 'Teacher_Quality_Low', 'Teacher_Quality_Medium', 'School_Type_Public', 'Peer_Influence_Neutral', 'Peer_Influence_Positive']

哑元编码后的数据已保存为：StudentPerformanceFactors_dummy_encoded.csv

哑元编码后的前5行数据：


,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score,Parental_Involvement_Low,Parental_Involvement_Medium,Access_to_Resources_Low,...,Teacher_Quality_Medium,School_Type_Public,Peer_Influence_Neutral,Peer_Influence_Positive,Learning_Disabilities_Yes,Parental_Education_Level_High School,Parental_Education_Level_Postgraduate,Distance_from_Home_Moderate,Distance_from_Home_Near,Gender_Male
0,23,84,7,73,0,3,67,1,0,0,...,1,1,0,1,0,1,0,0,1,1
1,19,64,8,59,2,4,61,1,0,0,...,1,1,0,0,0,0,0,1,0,0
2,24,98,7,91,2,4,74,0,1,0,...,1,1,1,0,0,0,1,0,1,1
3,29,89,8,98,1,4,71,1,0,0,...,1,1,0,0,0,1,0,1,0,1
4,19,92,6,65,3,4,70,0,1,0,...,0,1,1,0,0,0,0,0,1,0



第二题：多元线性回归建模
选取的自变量： ['Hours_Studied', 'Attendance', 'Previous_Scores', 'Tutoring_Sessions', 'Sleep_Hours', 'Motivation_Level', 'Parental_Involvement', 'Internet_Access', 'Learning_Disabilities']

回归方程（近似写法）：
Exam_Score = 41.972 + (0.292)*Hours_Studied + (0.199)*Attendance + (0.049)*Previous_Scores + (0.514)*Tutoring_Sessions - (0.029)*Sleep_Hours - (1.042)*Motivation_Level_Low - (0.540)*Motivation_Level_Medium - (1.956)*Parental_Involvement_Low - (1.056)*Parental_Involvement_Medium + (0.899)*Internet_Access_Yes - (0.881)*Learning_Disabilities_Yes

哑变量参考组说明：
- Motivation_Level 的参考组为 High
- Parental_Involvement 的参考组为 High
- Internet_Access 的参考组为 No
- Learning_Disabilities 的参考组为 No

模型评估指标：


,指标,数值
0,训练集 R^2,0.6342
1,测试集 R^2,0.6886
2,训练集 RMSE,2.3719
3,测试集 RMSE,2.0979


回归系数表：


...

系数显著性结果（OLS）：


,变量,系数,p值
0,const,42.0177,0.000000
1,Hours_Studied,0.2933,0.000000
2,Attendance,0.1982,0.000000
3,Previous_Scores,0.0489,0.000000
4,Tutoring_Sessions,0.5000,0.000000
5,Sleep_Hours,-0.0135,0.488884
6,Motivation_Level_Low,-1.0723,0.000000
7,Motivation_Level_Medium,-0.5301,0.000000
8,Parental_Involvement_Low,-1.9569,0.000000
9,Parental_Involvement_Medium,-1.0350,0.000000


总体 R^2: 0.6446
调整后 R^2: 0.6440
说明：测试集 R^2 越接近训练集 R^2，说明模型泛化能力越稳定。

第三题：模型解释与教育教学建议

一、模型优劣评价
1. 本模型的总体 R^2 为 0.6446，调整后 R^2 为 0.6440，说明模型能够解释约 64% 的考试成绩波动，具有较好的解释能力。
2. 训练集 R^2 = 0.6342，测试集 R^2 = 0.6886，两者较为接近，说明模型在新样本上的泛化表现较稳定。
3. 训练集 RMSE = 2.3719，测试集 RMSE = 2.0979，说明模型预测值与真实成绩之间的平均误差大约在 2 分左右。
4. 各自变量的 VIF 大多接近 1，最高约为 1.7479，表明模型不存在明显的多重共线性问题。
5. 模型的不足在于：线性回归只能刻画线性关系，且仍有约 35% 左右的成绩波动无法被当前变量解释，因此结果更适合用于解释和辅助预测，而不是绝对判断。

二、关键结果解释


,变量,系数,p值
8,Parental_Involvement_Low,-1.9569,0.0
6,Motivation_Level_Low,-1.0723,0.0
9,Parental_Involvement_Medium,-1.0350,0.0
11,Learning_Disabilities_Yes,-0.8793,0.0
10,Internet_Access_Yes,0.8441,0.0
7,Motivation_Level_Medium,-0.5301,0.0
4,Tutoring_Sessions,0.5000,0.0
1,Hours_Studied,0.2933,0.0
2,Attendance,0.1982,0.0
3,Previous_Scores,0.0489,0.0


1. Hours_Studied、Attendance、Tutoring_Sessions、Previous_Scores 的系数为正，说明学习投入和既有基础越好，考试成绩通常越高。
2. Internet_Access_Yes 的系数为正，说明具备网络条件的学生平均成绩更高，学习资源可及性较重要。
3. Motivation_Level_Low 和 Motivation_Level_Medium 的系数为负，说明相对于 High 组，学习动机较低的学生成绩更低。
4. Parental_Involvement_Low 和 Parental_Involvement_Medium 的系数为负，说明相对于家长高参与组，家长参与不足会降低学生成绩。
5. Learning_Disabilities_Yes 的系数为负，说明存在学习障碍的学生可能需要更多个性化支持。
6. Sleep_Hours 在本模型中的 p 值较大，未达到显著水平，说明它在线性模型中的单独影响不明显。

三、教育教学建议
1. 教师可重点关注学习时长和出勤率较低的学生，尽早进行学习习惯干预。
2. 对基础较弱的学生，可以增加针对性的课后辅导与阶段性反馈，帮助其稳步提升。
3. 学校和教师应通过目标激励、学习反馈和课堂参与设计，提高学生的学习动机。
4. 家校合作非常重要，应加强家长与学校之间的沟通，提升家长的教育参与程度。
5. 对网络条件不足或存在学习障碍的学生，应提供额外资源支持与差异化教学帮助。
6. 在实际教学中，可将本模型作为学生学习预警和教学干预的辅助工具，而不是唯一依据。


## AI对话记录
- AI对话链接：本次对话在当前 Codex 会话中完成。

### 第1轮
- 提问摘要：什么是哑元编码？第一题要求到底是在做什么？
- AI回答摘要：解释了哑元编码是把分类变量转换成多个 `0/1` 变量，以便进入线性回归模型；并结合 `Motivation_Level` 举例说明，第一题的本质是完成建模前的数据预处理。

### 第2轮
- 提问摘要：为什么原始 CSV 没有变化？能否导出一个新的哑元编码结果表？
- AI回答摘要：说明 Notebook 中生成的是新的数据框 `df_encoded`，不会覆盖原始 CSV；随后补充代码，将处理后的结果导出为 `StudentPerformanceFactors_dummy_encoded.csv`，方便查看编码结果。

### 第3轮
- 提问摘要：第二题应该选哪些自变量来预测 `Exam_Score`，以及线性模型准备怎么做？
- AI回答摘要：根据变量含义与数据情况，选择了 `Hours_Studied`、`Attendance`、`Previous_Scores`、`Tutoring_Sessions`、`Sleep_Hours`、`Motivation_Level`、`Parental_Involvement`、`Internet_Access`、`Learning_Disabilities` 等变量，使用多元线性回归建模，并通过 `R^2`、`RMSE`、`VIF` 和 `p` 值评价模型表现与变量作用。

### 第4轮
- 提问摘要：第三题如何解释模型结果，并给出作业所需的总结分析？
- AI回答摘要：补充了模型优劣、泛化能力、多重共线性、关键变量影响方向及教育教学建议的完整说明，并另外整理成 Markdown 分析文档，便于直接写入作业。